## Day 2 — Spark Architecture + Deeper Transformations

**Goal:**
- Understand Driver vs Executors
- See lazy execution in action
- Work with slightly more realistic data

### Cell 1 — Setup data

In [ ]:
data = [
    ("Alice",   "RO", 100),
    ("Bob",     "RO", 200),
    ("Charlie", "UK", 150),
    ("Diana",   "DE", 300),
    ("Eve",     "RO",  50),
    ("Frank",   "DE", 400),
    ("George",  "UK", 120),
    ("Helen",   "RO", 180),
]

columns = ["name", "country", "amount"]

df = spark.createDataFrame(data, columns)
df.show()

### Cell 2 — Basic exploration

Observe: data types and summary stats.

In [ ]:
df.printSchema()
df.describe().show()

### Cell 3 — Transformation chain

**Key idea — Lazy Evaluation:**  
Spark does NOT execute step-by-step like Python. It builds a plan, then runs everything at once when an action is called.

In [ ]:
from pyspark.sql.functions import col

df_transformed = (
    df
    .filter(col("amount") > 100)
    .withColumn("amount_double", col("amount") * 2)
    .select("name", "country", "amount_double")
)

df_transformed.show()

### Cell 4 — Prove lazy execution

In [ ]:
df_lazy = df.filter(col("amount") > 100)

# No action yet → nothing runs
print("Nothing executed yet!")

df_lazy.show()  # NOW it runs

### Cell 5 — Execution plan

Look for the **Logical plan** and **Physical plan**.  
You don't need to fully understand it yet — just get used to seeing it.

In [ ]:
df_transformed.explain(True)

### Cell 6 — Aggregations

In [ ]:
# Quick version
df.groupBy("country").agg({"amount": "sum"}).show()

# Cleaner version with alias
from pyspark.sql.functions import sum

df.groupBy("country").agg(sum("amount").alias("total_amount")).show()

### Cell 7 — Sorting results

In [ ]:
df.groupBy("country") \
  .agg(sum("amount").alias("total_amount")) \
  .orderBy(col("total_amount").desc()) \
  .show()

### Cell 8 — Concept: Driver vs Executors

```
Your Notebook (Driver)
        │
        ▼
   Spark Context
        │
   ┌────┴────┐
   ▼         ▼
Executor1  Executor2  ...  ExecutorN
(Worker)   (Worker)        (Worker)
```

- **Driver** → controls everything (your notebook)
- **Executors** → do the actual work (distributed)

When you call `.show()`:
1. Driver sends tasks
2. Executors process data in parallel
3. Results come back to Driver

### Cell 9 — Boolean column

In [ ]:
df2 = df.withColumn("high_value", col("amount") > 200)
df2.show()

### Cell 10 — Mini challenge

Write code to:
1. Filter only `country = "RO"`
2. Add a column: `amount_with_tax = amount * 1.19`
3. Show total per country

In [ ]:
# Your solution here


---

## Day 2 Checklist

- [ ] Created DataFrame and used `printSchema()` + `describe()`
- [ ] Chained `filter` → `withColumn` → `select` as one plan
- [ ] Confirmed lazy evaluation: `filter` alone doesn't run, `.show()` triggers it
- [ ] Read `explain(True)` output — spotted Logical and Physical plan
- [ ] Used `groupBy` + `agg` + `orderBy` together
- [ ] Understood Driver vs Executors
- [ ] Completed the mini challenge

**Common mistakes to avoid:**
- Thinking Spark runs line-by-line — it doesn't
- Ignoring `.explain()` — it's your debugging superpower
- Not chaining transformations — always chain before triggering an action

**Next:** Day 3 — Spark Runtime: Jobs, Stages, Tasks